In [1]:
import duckdb
import pandas as pd 
import numpy as np

In [2]:
con = duckdb.connect()

In [3]:
con.sql("""
SELECT COUNT(*) AS months, MIN(date) AS first_month, MAX(date) as last_month
FROM '../data/processed/national_monthly_registrations.parquet'
""").df()

,months,first_month,last_month
0,117,2017-01-01,2026-09-01


In [4]:
sql_data = con.sql("""
    with shares as(
    select brand, bev_ytd26, bev_ytd26 * 100.0 / sum(bev_ytd26) over () as bev_market_share
    from '../data/processed/brand_registrations.parquet'
    )
    select * from shares
    order by bev_ytd26 desc
""").df()

brands = pd.read_parquet('../data/processed/brand_registrations.parquet')

merged_df = brands.merge(
    sql_data,
    how='outer',
    on='brand',
    validate='one_to_one'
)

assert np.isclose(merged_df['bev_market_share'], merged_df['bev_market_share_ytd26']).all()
assert merged_df['brand'].count() == len(brands)

In [5]:
con.sql("""
create or replace temp view ranks as 
select brand, bev_ytd26, rank() over (order by bev_ytd26 desc) as bev_rank
from '../data/processed/brand_registrations.parquet'
""")

top_10 = con.sql("select * from ranks order by bev_rank limit 10").df()
bmw_rank = con.sql("select * from ranks where brand = 'bmw'").df()

In [6]:
pandas_rank = brands.set_index('brand')['bev_ytd26'].rank(ascending=False, method='min')
pandas_rank['bmw']


assert bmw_rank['bev_rank'].iloc[0] == pandas_rank['bmw']
assert set(top_10['brand']) == set(brands.nlargest(10, 'bev_ytd26')['brand'])

In [7]:
growth_df = con.sql(""" 
with market as(
select sum(total_ytd26) as total_26, sum(total_ytd25) as total_25, sum(bev_ytd26) as total_bev_26,
sum(bev_ytd25) as total_bev_25, sum(phev_ytd26) as total_phev_26, sum(phev_ytd25) as total_phev_25
from '../data/processed/brand_registrations.parquet'
),
bmw as(
select 
    total_ytd26, total_ytd25, bev_ytd26, 
    bev_ytd25, phev_ytd26, phev_ytd25
from '../data/processed/brand_registrations.parquet'
where brand = 'bmw'
)

select 
    (total_26 - total_25) * 100.0 / total_25 as market_total_growth,
    (total_bev_26 - total_bev_25) * 100.0 / total_bev_25 as market_bev_growth,
    (total_phev_26 - total_phev_25) * 100.0 / total_phev_25 as market_phev_growth,
    (total_ytd26 - total_ytd25) * 100.0 / total_ytd25 as bmw_total_growth,
    (bev_ytd26 - bev_ytd25) * 100.0 / bev_ytd25 as bmw_bev_growth,
    (phev_ytd26 - phev_ytd25) * 100.0 / phev_ytd25 as bmw_phev_growth
from market, bmw
""").df()

bmw_row = brands.set_index('brand').loc['bmw']

assert np.isclose(bmw_row['total_growth_ytd'], growth_df['bmw_total_growth'].iloc[0])
assert np.isclose(bmw_row['bev_growth_ytd'], growth_df['bmw_bev_growth'].iloc[0])
assert np.isclose(bmw_row['phev_growth_ytd'], growth_df['bmw_phev_growth'].iloc[0])

assert np.isclose(((brands['total_ytd26'].sum() - brands['total_ytd25'].sum()) * 100 / brands['total_ytd25'].sum()),growth_df['market_total_growth'].iloc[0])
assert np.isclose(((brands['bev_ytd26'].sum() - brands['bev_ytd25'].sum()) * 100 / brands['bev_ytd25'].sum()),growth_df['market_bev_growth'].iloc[0])
assert np.isclose(((brands['phev_ytd26'].sum() - brands['phev_ytd25'].sum()) * 100 / brands['phev_ytd25'].sum()),growth_df['market_phev_growth'].iloc[0])

In [8]:
charging_density = con.sql("""
with ports as(
select "Bundesland" as bundesland, sum("Anzahl Ladepunkte") as charging_ports
from '../data/processed/charging_stations_clean.parquet'
group by bundesland
)

select ports.bundesland, ports.charging_ports, a.area_in_sqkm, p.population, c.car_stock,
ports.charging_ports * 100000 / p.population as ports_per_100k,
ports.charging_ports * 10000 / c.car_stock as ports_per_10k_cars,
ports.charging_ports * 1000 / a.area_in_sqkm as ports_per_1000sqkm
from ports

join '../data/processed/area_by_bundesland.parquet' a on a.bundesland = ports.bundesland
join '../data/processed/population_by_bundesland.parquet' p on p.bundesland = ports.bundesland
join '../data/processed/car_stock_by_bundesland.parquet' c on c.bundesland = ports.bundesland
order by ports_per_100k desc
""").df()

assert len(charging_density) == 16
assert charging_density['charging_ports'].sum() == pd.read_parquet('../data/processed/charging_stations_clean.parquet')['Anzahl Ladepunkte'].sum()

In [9]:
rolling_df = con.sql("""
select date, 
bev,

bev - lag(bev) over(order by date) 
as change_vs_last_month,

lag(bev, 12) over(order by date) 
as last_year,

(bev - last_year) * 100.0 / nullif(last_year, 0) 
as growth_same_month_last_year,

case 
    when row_number() over(order by date) >= 12 
    then sum(bev) over (order by date rows between 11 preceding and current row)
end as bev_rolling_12m

from '../data/processed/national_monthly_registrations.parquet'
order by date


""").df()

In [10]:
pandas_check = pd.read_parquet('../data/processed/national_monthly_registrations.parquet')

pandas_check = pandas_check.sort_values('date').reset_index(drop=True)

assert np.isclose(pandas_check['bev'].diff(), rolling_df['change_vs_last_month'], equal_nan=True).all()
assert np.isclose((pandas_check['bev'] - pandas_check['bev'].shift(12)) * 100 / pandas_check['bev'].shift(12), rolling_df['growth_same_month_last_year'], equal_nan=True).all()
assert np.isclose(pandas_check['bev'].rolling(12).sum(), rolling_df['bev_rolling_12m'], equal_nan=True).all()

In [11]:
state_ranks = con.sql("""
with ranks as( 
select 
bundesland,
rank() over(order by bev_share_ytd26 desc) as bev_share_rank,
rank() over(order by phev_share_ytd26 desc) as phev_share_rank
from '../data/processed/regional_registrations.parquet'
)
select ranks.bundesland, ranks.bev_share_rank, ranks.phev_share_rank,
d.ports_per_100k, d.ports_per_10k_cars
from ranks
join charging_density d
on ranks.bundesland = d.bundesland
order by bev_share_rank

""").df()

In [12]:
state_ranks_pandas = pd.read_parquet('../data/processed/regional_registrations.parquet')
pandas_ranks = state_ranks_pandas.set_index('bundesland')[['bev_share_ytd26', 'phev_share_ytd26']].rank(ascending=False, method='min')
sql_ranks = state_ranks.set_index('bundesland')[['bev_share_rank', 'phev_share_rank']]

assert pandas_ranks['bev_share_ytd26'].sort_index().equals(sql_ranks['bev_share_rank'].sort_index().astype(float))
assert pandas_ranks['phev_share_ytd26'].sort_index().equals(sql_ranks['phev_share_rank'].sort_index().astype(float))

In [13]:
corr_sql = con.sql("""
select corr(r.phev_share_ytd26, d.ports_per_10k_cars) as phev_ports_corr
from '../data/processed/regional_registrations.parquet' r
join charging_density d
on r.bundesland = d.bundesland
""").df()

corr_merged = state_ranks_pandas.merge(charging_density, on='bundesland', how='inner', validate='one_to_one')
assert len(corr_merged) == len(state_ranks_pandas)

corr_pandas = corr_merged['phev_share_ytd26'].corr(corr_merged['ports_per_10k_cars'])

assert np.isclose(corr_sql['phev_ports_corr'].iloc[0], corr_pandas)
corr_sql

,phev_ports_corr
0,0.835733


### Summary: SQL checks

This notebook recomputes the project's key figures in SQL (DuckDB, querying the Parquet files in `data/processed/` directly) and checks each result against pandas.

| # | Question | SQL used |
|---|---|---|
| 1 | BEV market share per brand, Jan–Sep 2026 | CTE, window `SUM() OVER ()` |
| 2 | Top 10 brands by BEV registrations; BMW's rank | `RANK()`, temporary view |
| 3 | BMW vs market growth, total / BEV / PHEV | two CTEs, aggregation |
| 4 | Charging ports per 100,000 inhabitants, per 10,000 cars and per 1,000 km² by state | `GROUP BY`, join of four tables |
| 5 | Monthly BEV change vs last month and last year; rolling 12-month total | `LAG()`, window frame, `CASE` |
| 6 | State ranking by BEV and PHEV share, joined with charging density; correlation of PHEV share with ports per 10,000 cars | `RANK()`, join with a pandas DataFrame, `corr()` |

**Result:** every SQL result matches the pandas calculation (asserted in each cell), so the headline figures in the README are reproduced independently in SQL.

**Notes:**
- DuckDB reads the Parquet files in place with an in-memory connection, so no database file is created.
- The correlation in query 6 uses all charging ports and all 16 states. It differs from the README's r = 0.73, which uses AC-only ports and the 13 territorial states.